# 06 — Combined results
Run notebooks 00 through 05 first. Load saved tree, GRU, LSTM, and baseline metrics and predictions without retraining. Results not yet generated remain blank. Tree model selection remains unchanged.


In [ ]:
%pip -q install numpy pandas joblib matplotlib


In [ ]:
from pathlib import Path
from google.colab import drive
from IPython.display import display
import joblib
import numpy as np
import pandas as pd

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/KC_CO2_Modular_Notebooks")
RUN_DIR = PROJECT / "runs" / "local_sessions_5_10_all_approaches_v1"
import matplotlib.pyplot as plt


In [ ]:
result_path = RUN_DIR / "tree_results.joblib"
metadata_path = RUN_DIR / "preparation_metadata.joblib"
if not result_path.is_file() or not metadata_path.is_file():
    raise FileNotFoundError("Run notebooks 00, 01 and 02 first.")
saved = joblib.load(result_path)
metadata = joblib.load(metadata_path)
if saved["metadata"] != metadata:
    raise ValueError("Saved results do not match preparation. Rerun 01 and 02.")
if metadata.get("schema_version") != 2 or metadata["horizons"] != [5, 10] or metadata["approaches"] != ["direct", "recursive"]:
    raise ValueError("Results must contain both horizons and both approaches.")
metrics = saved["metrics"].copy()
predictions = saved["predictions"].copy()
keys = ["Horizon_min", "Approach", "Model", "Split"]
expected = {(h,a,m,s) for h in (5,10) for a in ("direct","recursive") for m in ("Random Forest","XGBoost") for s in ("Validation","Test")}
if metrics.duplicated(keys).any() or set(metrics[keys].itertuples(index=False, name=None)) != expected:
    raise ValueError("Expected one RF/XGBoost result for each horizon, approach and split.")
if not np.isfinite(metrics[["MAE", "RMSE", "R2"]].to_numpy(dtype=float)).all():
    raise ValueError("Model results must contain finite MAE, RMSE and R2 values.")
for horizon in (5,10):
    for approach in ("direct","recursive"):
        ranked = metrics.loc[metrics.Horizon_min.eq(horizon) & metrics.Approach.eq(approach) & metrics.Split.eq("Validation")].sort_values("Validation Rank")
        winner = saved["selected_models"][f"{horizon}min_{approach}"]
        if ranked.iloc[0].Model != winner:
            raise ValueError("Saved selection differs from validation ranking.")
print("Preparation:", metadata["run_id"])


## Model results


In [ ]:
combined_results = metrics.copy()
combined_results["Status"] = "Complete"
additional_results = pd.DataFrame(columns=combined_results.columns)


## GRU


In [ ]:
additional_prediction_frames = []

def load_model_metrics(model_name, model_names=None, result_path=None):
    model_names = model_names or {model_name}
    result_path = result_path or RUN_DIR / model_name / f"{model_name.lower()}_results.joblib"
    if not result_path.is_file():
        return pd.DataFrame(columns=combined_results.columns)
    saved_neural = joblib.load(result_path)
    saved_metadata = saved_neural.get("metadata", {})
    if any(saved_metadata.get(key) != value for key, value in metadata.items()):
        raise ValueError(f"{model_name} preparation metadata does not match the tree run.")
    metrics_frame = saved_neural.get("metrics")
    required = {"Horizon_min", "Approach", "Model", "Split", "MAE", "RMSE", "R2"}
    if metrics_frame is None or not required.issubset(metrics_frame.columns):
        raise ValueError(f"{model_name} metrics artifact has an incompatible schema.")
    if set(metrics_frame["Model"]) != set(model_names) or metrics_frame.duplicated(keys).any():
        raise ValueError(f"{model_name} contains duplicate or mismatched result rows.")
    expected_neural = {
        (horizon, approach, name, split)
        for name in model_names
        for horizon in (5, 10)
        for approach in ("direct", "recursive")
        for split in ("Validation", "Test")
    }
    actual_neural = set(metrics_frame[keys].itertuples(index=False, name=None))
    if actual_neural != expected_neural:
        raise ValueError(f"{model_name} artifact is incomplete: expected all four configurations.")
    if not np.isfinite(metrics_frame[["MAE", "RMSE", "R2"]].to_numpy(dtype=float)).all():
        raise ValueError(f"{model_name} contains non-finite metrics.")
    metrics_frame = metrics_frame.copy()
    metrics_frame["Status"] = "Complete"
    prediction_frame = saved_neural.get("predictions")
    prediction_columns = {"session_id", "time", "Actual", "Predicted", "Model", "Split", "Horizon_min", "Approach"}
    if prediction_frame is None or not prediction_columns.issubset(prediction_frame.columns):
        raise ValueError(f"{model_name} predictions artifact has an incompatible schema.")
    if set(prediction_frame["Model"]) != set(model_names):
        raise ValueError(f"{model_name} predictions contain mismatched models.")
    if not np.isfinite(prediction_frame[["Actual", "Predicted"]].to_numpy(dtype=float)).all():
        raise ValueError(f"{model_name} predictions contain non-finite values.")
    additional_prediction_frames.append(prediction_frame.copy())
    return metrics_frame

gru_results = load_model_metrics("GRU")

## LSTM


In [ ]:
lstm_results = load_model_metrics("LSTM")
baseline_metrics = load_model_metrics("Baselines", {"Persistence", "Linear Regression"}, RUN_DIR / "baseline_results" / "baseline_results.joblib")
additional_results = pd.concat([gru_results, lstm_results, baseline_metrics], ignore_index=True)
if additional_prediction_frames:
    predictions = pd.concat([predictions, *additional_prediction_frames], ignore_index=True)


In [ ]:
if not additional_results.empty:
    if not set(additional_results.Model).issubset({"GRU", "LSTM", "Persistence", "Linear Regression"}) or additional_results.duplicated(keys).any():
        raise ValueError("Additional model results require unique model/horizon/approach/split rows.")
    combined_results = pd.concat([combined_results, additional_results], ignore_index=True)
missing = []
for horizon in (5, 10):
    for approach in ("direct", "recursive"):
        for model in ("GRU", "LSTM", "Persistence", "Linear Regression" ):
            for split in ("Validation", "Test"):
                exists = combined_results.Horizon_min.eq(horizon) & combined_results.Approach.eq(approach) & combined_results.Model.eq(model) & combined_results.Split.eq(split)
                if not exists.any():
                    missing.append({"Horizon_min": horizon, "Approach": approach, "Model": model, "Split": split, "Status": "Not available"})
combined_results = pd.concat([combined_results, pd.DataFrame(missing)], ignore_index=True)
columns = ["Horizon_min", "Approach", "Model", "Split", "MAE", "RMSE", "R2", "Validation Rank", "Selected on validation", "Status"]
combined_results = combined_results[columns].sort_values(["Horizon_min", "Approach", "Split", "Validation Rank"], na_position="last")
for horizon in (5, 10):
    for approach in ("direct", "recursive"):
        print(f"\n{horizon}-minute {approach} — MAE/RMSE in ppm; R2 is dimensionless")
        table = combined_results.loc[combined_results.Horizon_min.eq(horizon) & combined_results.Approach.eq(approach)]
        print(table.round(4).fillna("").to_string(index=False))


## Metric comparisons


In [ ]:
available = combined_results.loc[combined_results.Status.eq("Complete")]
for horizon in (5, 10):
    for approach in ("direct", "recursive"):
        subset = available.loc[available.Horizon_min.eq(horizon) & available.Approach.eq(approach)]
        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
        for ax, metric in zip(axes, ("MAE", "RMSE", "R2")):
            subset.pivot(index="Model", columns="Split", values=metric).plot.bar(ax=ax, rot=0)
            ax.set(title=metric, ylabel="ppm" if metric != "R2" else "R²", xlabel="")
        fig.suptitle(f"{horizon}-minute {approach}")
        plt.tight_layout()
        plt.show()


## Saved test predictions


In [ ]:
for horizon in (5, 10):
    for approach in ("direct", "recursive"):
        fig, axes = plt.subplots(3, 2, figsize=(12, 15))
        for ax, model in zip(axes.flat, ("Random Forest", "XGBoost", "GRU", "LSTM", "Persistence", "Linear Regression")):
            frame = predictions.loc[predictions.Horizon_min.eq(horizon) & predictions.Approach.eq(approach) & predictions.Split.eq("Test") & predictions.Model.eq(model)]
            if frame.empty:
                ax.set_title(f"{model}: Not available")
                ax.set_axis_off()
                continue
            ax.scatter(frame.Actual, frame.Predicted, alpha=0.4)
            low = min(frame.Actual.min(), frame.Predicted.min())
            high = max(frame.Actual.max(), frame.Predicted.max())
            ax.plot([low, high], [low, high], "--", color="black")
            ax.set(title=model, xlabel="Actual future CO₂ (ppm)", ylabel="Predicted future CO₂ (ppm)")
        fig.suptitle(f"{horizon}-minute {approach}: test predictions")
        plt.tight_layout()
        plt.show()


## Tree feature importance


In [ ]:
feature_importance = saved["feature_importance"]
if not feature_importance.empty:
    for (horizon, approach, model), frame in feature_importance.groupby(["Horizon_min", "Approach", "Model"], sort=True):
        top = frame.nlargest(20, "Importance").sort_values("Importance")
        top.plot.barh(x="Feature", y="Importance", legend=False, figsize=(8, 6), title=f"{model}: {horizon}-minute {approach}")
        plt.tight_layout()
        plt.show()


## Export comparison


In [ ]:
combined_results.to_csv(RUN_DIR / "combined_results.csv", index=False)
print("Saved:", RUN_DIR / "combined_results.csv")
